In [34]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [35]:
# ============================
# 0. Paths and basic config
# ============================
import os
import torch
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel

# Use pretrained Qwen3-4B (base model)
MODEL_NAME = "Qwen/Qwen3-4B"

# Optional output directory for evaluation results
OUTPUT_DIR = "/content/drive/MyDrive/qwen3_4b_pretrained_eval-task-specific-prompt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_CSV_PATH = "/content/drive/MyDrive/test.csv"
TRAIN_CSV_PATH = "/content/drive/MyDrive/train.csv"

MAX_SEQ_LENGTH = 2048
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
# ============================
# 1. Load pretrained Qwen3-4B
# ============================
MODEL_NAME = "Qwen/Qwen3-4B"

print("Loading pretrained model:", MODEL_NAME)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,          # 4 bit quantization
    dtype          = torch.float16, # optional, Unsloth may pick this automatically
)

FastLanguageModel.for_inference(model)

print("Model and tokenizer loaded.")


Loading pretrained model: Qwen/Qwen3-4B
==((====))==  Unsloth 2025.11.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model and tokenizer loaded.


In [38]:
# ============================
# 2. Load train & test data, display heads, and compare label sets
# ============================

print("Loading train data from:", TRAIN_CSV_PATH)
train_df = pd.read_csv(TRAIN_CSV_PATH)

print("Loading test data from:", TEST_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

# Keep only needed columns and drop missing rows
train_df = train_df[["case_id", "input_finding", "output_disease"]].dropna()
test_df  = test_df[["case_id", "input_finding", "output_disease"]].dropna()

print("\nNumber of TRAIN rows:", len(train_df))
print("Number of TEST rows:", len(test_df))

print("\n--- TRAIN HEAD ---")
display(train_df.head())

print("\n--- TEST HEAD ---")
display(test_df.head())

# ------------------------------------------------
# Extract unique labels from TRAIN
# ------------------------------------------------
train_labels = (
    train_df["output_disease"]
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)
train_labels = train_labels[train_labels != ""]
train_label_set = set(train_labels.unique())

# ------------------------------------------------
# Extract unique labels from TEST
# ------------------------------------------------
test_labels = (
    test_df["output_disease"]
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)
test_labels = test_labels[test_labels != ""]
test_label_set = set(test_labels.unique())

# ------------------------------------------------
# Compare label spaces
# ------------------------------------------------
missing_in_train = test_label_set - train_label_set
missing_in_test  = train_label_set - test_label_set

print("\n======================")
print(" LABEL SPACE SUMMARY")
print("======================")
print(f"Unique TRAIN labels: {len(train_label_set)}")
print(f"Unique TEST labels:  {len(test_label_set)}")
print("----------------------")
print(f"Labels in TEST but NOT in TRAIN: {len(missing_in_train)}")
print(missing_in_train if missing_in_train else "None")
print("----------------------")
print(f"Labels in TRAIN but NOT in TEST: {len(missing_in_test)}")
print(missing_in_test if missing_in_test else "None")

# ------------------------------------------------
# Build allowed_labels strictly from TRAIN
# ------------------------------------------------
allowed_labels = ", ".join(sorted(train_label_set))
print("\nAllowed label list will be built from TRAIN ONLY.")
print("Number of allowed labels:", len(train_label_set))
# print("Preview:", allowed_labels[:300], "..." if len(allowed_labels) > 300 else "")
print(allowed_labels)


Loading train data from: /content/drive/MyDrive/train.csv
Loading test data from: /content/drive/MyDrive/test.csv

Number of TRAIN rows: 1236
Number of TEST rows: 386

--- TRAIN HEAD ---


,case_id,input_finding,output_disease
0,1,The liver surface is regular with no apparent ...,Liver lesion
1,2,"The liver and spleen are in normal position, s...","Kidney stone, Pancreatitis"
2,3,"The liver surface is smooth, and the size and ...","Liver cyst, Renal cyst, Splenic cyst"
3,4,"Abnormal liver morphology, with arc-shaped low...","Liver lesion, Liver cyst, Gallstone, Ascites, ..."
4,5,"Two round low-density lesions, approximately 5...","Liver lesion, Splenomegaly, Renal mass, Colore..."



--- TEST HEAD ---


,case_id,input_finding,output_disease
0,1,The liver is normal in size and shape with hom...,"Renal cyst, Adrenal hyperplasia, Adrenal calci..."
1,2,"The liver is normal in position, size, and sha...","Liver lesion, Atherosclerosis"
2,3,"The liver surface is smooth, with coordinated ...","Liver cyst, Pancreas calcifications, Accessory..."
3,4,The liver surface is smooth with coordinated s...,"Cholecystitis, Renal cyst, Ascites"
4,5,"The surface of the liver is smooth, and the si...","Liver calcifications, Gallstone"



 LABEL SPACE SUMMARY
Unique TRAIN labels: 84
Unique TEST labels:  68
----------------------
Labels in TEST but NOT in TRAIN: 19
{'Gastric cancer (possible)', 'Bilateral polycystic kidneys', 'Leiomyoma', 'the most appropriate disease labels based on the findings are: Emphysematous pyelonephritis', 'Mesenchymal tumor', 'Ectopic pancreas', 'Left renal infection lesion with renal pneumaturia and perinephric inflammation exudation in the left kidney corresponds to a kidney infection', 'which is a severe form of kidney infection. The findings also indicate gallbladder stones', 'Teratoma', 'Liver calcifications', 'Duodenal diverticulum', 'which is not explicitly listed in the candidate disease list. However', 'the presence of gas in the kidney suggests emphysematous pyelonephritis', 'Pancreas calcifications', 'which correspond to "Gallstone" in the candidate disease list.\n\nTherefore', 'Gallstone.', 'Retroperitoneal cyst', 'Colorectal cancer (possible)', 'Gastrointestinal stromal tumor'}
--

In [39]:
# ============================
# 3. System prompt (same style as training)
# ============================

# system_prompt = (
#     "You are a clinical Named Entity Recognition (NER) and multi-label classification model. "
#     "Read the abdominal radiology findings and identify all diseases that are present. "
#     "Use only disease names from the allowed disease label list. "
#     "Return the diseases as a comma separated list using the exact wording from the list. "
#     "If none apply, output: No acute abnormality. "
#     f"Allowed disease label list: {allowed_labels}."
# )


system_prompt = f"""
You are an expert radiology NLP model for abdominal CT reports.

Task:
Given the finding section of an abdominal CT report, identify all disease labels that are explicitly supported by the text.

Rules:
1. Use only disease names from the allowed disease label list below.
2. Treat this as a multi label task and output all applicable diseases, not just one.
3. Base labels only on abnormalities that are clearly present in the finding. Ignore structures described as normal or without abnormality.
4. Do not invent new labels or synonyms. Use the exact wording from the allowed list.
5. If no disease label applies, output exactly: No acute abnormality

Typical patterns in this dataset:
- Cystic or low density round lesions in liver, kidney, spleen, or pancreas usually correspond to the appropriate cyst labels when present in the list.
- Stones or calculi in kidney, ureter, gallbladder, or bile ducts correspond to stone labels when present in the list.
- Dilatation of the renal pelvis and calyces with or without ureteral dilatation corresponds to Hydronephrosis when that label exists.
- Diffuse low attenuation of the liver with fatty change language corresponds to Fatty liver.
- Enlarged spleen corresponds to Splenomegaly when that label exists.
- Vascular wall calcifications correspond to Atherosclerosis when that label exists.

Allowed disease label list:
{allowed_labels}
"""

In [40]:
# ============================
# 4. Inference helper
# ============================
def predict_diseases(finding_text: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                "Clinical findings:\n"
                f"{finding_text}\n\n"
                "List all diseases present using ONLY labels from the allowed disease list. "
                "Separate multiple diseases with commas. If none apply, output: No acute abnormality"
            ),
        },
    ]

    # Build model input
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,     # base model cannot do thinking mode
    )

    # Move tensors
    if isinstance(inputs, dict):
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        input_ids = inputs["input_ids"]
        attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids))
    else:
        inputs = inputs.to(DEVICE)
        input_ids = inputs
        attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean leftover metadata
    if "</think>" in text:
        text = text.split("</think>")[-1]

    if "assistant" in text:
        text = text.split("assistant")[-1]

    return text.strip()


In [41]:
# ============================
# 5. Run on ALL test samples
# ============================

results = []   # list of rows for CSV

print("Running predictions on", len(test_df), "samples...")

for idx, row in test_df.iterrows():

    # -------------------------
    # Use case_id from test_df
    # -------------------------
    case_id = row["case_id"]

    # Extract text and labels
    finding  = str(row["input_finding"])   # ensure string, avoid NaN issues
    gold_raw = row["output_disease"]
    pred_raw = predict_diseases(finding)

    # -----------------------------
    # Normalize ground truth labels
    # -----------------------------
    gold_list = [g.strip() for g in str(gold_raw).split(",") if g.strip()]
    gold_list_sorted = sorted(gold_list)
    gold_sorted_str = ", ".join(gold_list_sorted)

    # -------------------------
    # Normalize predicted labels
    # -------------------------
    pred_list = [p.strip() for p in str(pred_raw).split(",") if p.strip()]
    pred_list_sorted = sorted(pred_list)
    pred_sorted_str = ", ".join(pred_list_sorted)

    # -------------------------
    # Store row in results
    # -------------------------
    results.append({
        "case_id": case_id,
        "output_diseases": gold_sorted_str,
        "input_finding": finding,
        "predicted_disease": pred_sorted_str,
    })

    # Optional: print progress
    print("=" * 80)
    print(f"Index: {idx} / {len(test_df) - 1}")
    print("Case:", case_id)
    print("Gold:", gold_sorted_str)
    print("Pred:", pred_sorted_str)


# ============================
# 6. Save sorted results to CSV
# ============================

output_csv = OUTPUT_DIR + ".csv"
pd.DataFrame(results).to_csv(output_csv, index=False)

print("Saved prediction results to:", output_csv)


Running predictions on 386 samples...
Index: 0 / 385
Case: 1
Gold: Adrenal calcification, Adrenal hyperplasia, Renal cyst
Pred: Adrenal nodule, Kidney infarction, Liver lesion, No acute abnormality, Renal mass
Index: 1 / 385
Case: 2
Gold: Atherosclerosis, Liver lesion
Pred: Abdominal mass, No acute abnormality, Splenomegaly
Index: 2 / 385
Case: 3
Gold: Accessory spleen, Atherosclerosis, Gastric cancer (possible), Liver cyst, Pancreas calcifications, Renal cyst
Pred: Fatty liver, Kidney stone, Liver lesion, Pancreatic calcification, Splenic mass, Splenomegaly
Index: 3 / 385
Case: 4
Gold: Ascites, Cholecystitis, Renal cyst
Pred: Ascites, Fatty liver, Kidney stone, No acute abnormality
Index: 4 / 385
Case: 5
Gold: Gallstone, Liver calcifications
Pred: Atherosclerosis, Liver calcification, No acute abnormality
Index: 5 / 385
Case: 6
Gold: Adrenal mass, Bile duct dilatation
Pred: Abdominal mass, Bile duct dilatation, Gas in the biliary tree, No acute abnormality, Renal mass, Splenomegaly
In

In [42]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
)


# -----------------------------
# 1. LABEL PROCESSING
# -----------------------------
def process_labels(text):
    """Convert comma separated string into a clean, deduplicated, sorted label list."""
    if not isinstance(text, str) or text.strip() == "":
        return []
    labels = [l.strip() for l in text.split(",") if l.strip()]
    # deduplicate and sort for canonical form
    return sorted(set(labels))


# -----------------------------
# 2. SIMPLE LABEL LEVEL BLEU AND ROUGE (SET BASED)
# -----------------------------
def label_bleu(pred_labels, true_labels):
    """
    Simple BLEU style score over label sets:
    BLEU_label = |intersection| / |pred|
    Uses sets, order independent, lists should be sorted upstream for consistency.
    """
    pred_set = set(pred_labels)
    true_set = set(true_labels)
    if len(pred_set) == 0:
        return 1.0 if len(true_set) == 0 else 0.0
    inter = len(pred_set & true_set)
    return inter / len(pred_set)


def label_rouge(pred_labels, true_labels):
    """
    Simple ROUGE style score over label sets:
    ROUGE_label = |intersection| / |true|
    Uses sets, order independent, lists should be sorted upstream for consistency.
    """
    pred_set = set(pred_labels)
    true_set = set(true_labels)
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0
    inter = len(pred_set & true_set)
    return inter / len(true_set)


# -----------------------------
# 3. GLOBAL METRICS
# -----------------------------
def compute_global_metrics(y_true_bin, y_pred_bin, y_true_lists, y_pred_lists):
    """Compute and return global multi label metrics, including Hamming and over or under prediction rates."""

    # Exact match and Jaccard (sample-based)
    jaccard = jaccard_score(y_true_bin, y_pred_bin, average="samples")
    exact_match = accuracy_score(y_true_bin, y_pred_bin)

    # Micro and Macro metrics
    micro_prec = precision_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    micro_rec = recall_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)
    micro_f1 = f1_score(y_true_bin, y_pred_bin, average="micro", zero_division=0)

    macro_prec = precision_score(y_true_bin, y_pred_bin, average="macro", zero_division=0)
    macro_rec = recall_score(y_true_bin, y_pred_bin, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true_bin, y_pred_bin, average="macro", zero_division=0)

    # Hamming loss (fraction of misclassified label decisions)
    hamming = hamming_loss(y_true_bin, y_pred_bin)

    # Over and under prediction statistics
    total_fp = 0
    total_fn = 0
    total_pred_labels = 0
    total_true_labels = 0
    total_card_diff_abs = 0

    for gt_labels, pred_labels in zip(y_true_lists, y_pred_lists):
        set_gt = set(gt_labels)
        set_pred = set(pred_labels)

        fp_labels = set_pred - set_gt
        fn_labels = set_gt - set_pred

        total_fp += len(fp_labels)
        total_fn += len(fn_labels)
        total_pred_labels += len(set_pred)
        total_true_labels += len(set_gt)
        total_card_diff_abs += abs(len(set_pred) - len(set_gt))

    over_pred_rate = (total_fp / total_pred_labels) if total_pred_labels > 0 else 0.0
    under_pred_rate = (total_fn / total_true_labels) if total_true_labels > 0 else 0.0
    mean_card_mismatch = (
        total_card_diff_abs / len(y_true_lists) if len(y_true_lists) > 0 else 0.0
    )

    print("\n" + "=" * 40)
    print("       PERFORMANCE METRICS")
    print("=" * 40)
    print(f"Exact Match Accuracy:  {exact_match:.4f}")
    print(f"Jaccard Score (IoU):   {jaccard:.4f}")
    print("-" * 40)
    print(f"Micro Precision:       {micro_prec:.4f}")
    print(f"Micro Recall:          {micro_rec:.4f}")
    print(f"Micro F1 Score:        {micro_f1:.4f}")
    print("-" * 40)
    print(f"Macro Precision:       {macro_prec:.4f}")
    print(f"Macro Recall:          {macro_rec:.4f}")
    print(f"Macro F1 Score:        {macro_f1:.4f}")
    print("-" * 40)
    print(f"Hamming Loss:          {hamming:.4f}")
    print(f"Over prediction rate:  {over_pred_rate:.4f}")
    print(f"Under prediction rate: {under_pred_rate:.4f}")
    print(f"Mean |#pred-#true|:    {mean_card_mismatch:.4f}")
    print("=" * 40)

    return {
        "exact_match": exact_match,
        "jaccard": jaccard,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
        "micro_f1": micro_f1,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1,
        "hamming_loss": hamming,
        "over_prediction_rate": over_pred_rate,
        "under_prediction_rate": under_pred_rate,
        "mean_label_cardinality_mismatch": mean_card_mismatch,
    }


# -----------------------------
# 4. TRAIN BASED REGION DEFINITION
# -----------------------------
def compute_regions_from_train(train_df):
    """
    Split labels into minority, middle, majority groups
    based on TRAIN SET label frequencies.
    """
    labels = (
        train_df["output_disease"]
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    labels = labels[labels != ""]
    label_counts = labels.value_counts()

    sorted_labels = label_counts.sort_values()
    n = len(sorted_labels)
    third = max(1, n // 3)

    minority = sorted_labels.index[:third].tolist()
    middle = sorted_labels.index[third:-third].tolist() if n > 2 * third else []
    majority = sorted_labels.index[-third:].tolist()

    print("\nTRAIN label frequency regions:")
    print(f"  Minority labels: {len(minority)}")
    print(f"  Middle labels:   {len(middle)}")
    print(f"  Majority labels: {len(majority)}")

    return minority, middle, majority


def compute_region_metrics(y_true_bin, y_pred_bin, mlb, regions):
    """Compute regional metrics given regions defined on TRAIN labels."""
    print("\n" + "=" * 40)
    print(" PERFORMANCE BY LABEL FREQUENCY REGION (TRAIN BASED)")
    print("=" * 40)

    results = []

    label_to_idx = {lbl: i for i, lbl in enumerate(mlb.classes_)}

    for region_name, label_list in regions.items():
        if len(label_list) == 0:
            print(f"\n[{region_name.upper()}] No labels in this region.")
            continue

        idx = [label_to_idx[l] for l in label_list if l in label_to_idx]

        if len(idx) == 0:
            print(f"\n[{region_name.upper()}] No overlapping labels between TRAIN region and TEST set.")
            continue

        idx = np.array(idx, dtype=int)
        y_true_region = y_true_bin[:, idx]
        y_pred_region = y_pred_bin[:, idx]

        micro_f1 = f1_score(y_true_region, y_pred_region, average="micro", zero_division=0)
        macro_f1 = f1_score(y_true_region, y_pred_region, average="macro", zero_division=0)

        print(f"\n[{region_name.upper()}] ({len(idx)} labels)")
        print(f"  Micro F1: {micro_f1:.4f}")
        print(f"  Macro F1: {macro_f1:.4f}")

        results.append({
            "region": region_name,
            "num_labels": len(idx),
            "micro_f1": micro_f1,
            "macro_f1": macro_f1,
        })

    return results


# -----------------------------
# 5. CARDINALITY BASED METRICS (BY NUM TRUE DISEASES)
# -----------------------------
def compute_cardinality_metrics(sample_rows):
    """
    Analyze performance as a function of the number of ground truth labels.

    For each distinct num_gt_labels, compute:
      - count of samples
      - mean Jaccard
      - exact match rate
      - mean BLEU label
      - mean ROUGE label
      - mean number of FN and FP labels
      - mean over and under prediction rate
      - distribution of error types
    """
    df_s = pd.DataFrame(sample_rows)

    print("\n" + "=" * 40)
    print(" PERFORMANCE BY NUMBER OF TRUE DISEASE LABELS")
    print("=" * 40)

    cardinality_metrics = []

    for num_labels, group in df_s.groupby("num_gt_labels"):
        count = len(group)
        mean_jaccard = group["jaccard_score"].mean()
        exact_match_rate = group["exact_match"].mean()
        mean_bleu = group["bleu_label"].mean()
        mean_rouge = group["rouge_label"].mean()
        mean_fn = group["num_fn_labels"].mean()
        mean_fp = group["num_fp_labels"].mean()
        mean_over_rate = group["over_pred_rate_sample"].mean()
        mean_under_rate = group["under_pred_rate_sample"].mean()

        # error type distribution
        error_dist = group["error_type"].value_counts(normalize=True)

        print(f"\n[Num true labels = {num_labels}]")
        print(f"  Samples:              {count}")
        print(f"  Mean Jaccard:         {mean_jaccard:.4f}")
        print(f"  Exact match rate:     {exact_match_rate:.4f}")
        print(f"  Mean BLEU label:      {mean_bleu:.4f}")
        print(f"  Mean ROUGE label:     {mean_rouge:.4f}")
        print(f"  Mean #FN labels:      {mean_fn:.4f}")
        print(f"  Mean #FP labels:      {mean_fp:.4f}")
        print(f"  Mean over pred rate:  {mean_over_rate:.4f}")
        print(f"  Mean under pred rate: {mean_under_rate:.4f}")
        print("  Error type fractions:")
        for etype, frac in error_dist.items():
            print(f"    {etype}: {frac:.4f}")

        metrics = {
            "count": count,
            "mean_jaccard": mean_jaccard,
            "exact_match_rate": exact_match_rate,
            "mean_bleu_label": mean_bleu,
            "mean_rouge_label": mean_rouge,
            "mean_num_fn_labels": mean_fn,
            "mean_num_fp_labels": mean_fp,
            "mean_over_pred_rate_sample": mean_over_rate,
            "mean_under_pred_rate_sample": mean_under_rate,
        }

        # add error type distribution as separate metrics
        for etype, frac in error_dist.items():
            metrics[f"error_ratio_{etype}"] = frac

        cardinality_metrics.append({
            "num_gt_labels": int(num_labels),
            "metrics": metrics,
        })

    return cardinality_metrics


# -----------------------------
# 6. MAIN EVALUATION PIPELINE
# -----------------------------
def evaluate_multilabel_predictions(prediction_csv, train_csv):
    print("Loading prediction file:", prediction_csv)
    df = pd.read_csv(prediction_csv)

    print("Loading train file:", train_csv)
    train_df = pd.read_csv(train_csv)
    train_df = train_df[["case_id", "input_finding", "output_disease"]].dropna()

    # Prepare labels (sorted)
    if "output_diseases" in df.columns:
        true_col = "output_diseases"
    elif "output_disease" in df.columns:
        true_col = "output_disease"
    else:
        raise ValueError("Expected 'output_diseases' or 'output_disease' in predictions CSV.")

    if "predicted_disease" not in df.columns:
        raise ValueError("Expected 'predicted_disease' column in predictions CSV.")

    df["y_true"] = df[true_col].apply(process_labels)
    df["y_pred"] = df["predicted_disease"].apply(process_labels)

    # Lists are sorted and deduplicated now
    y_true_lists = df["y_true"].tolist()
    y_pred_lists = df["y_pred"].tolist()

    # One hot encoding
    mlb = MultiLabelBinarizer()
    mlb.fit(y_true_lists + y_pred_lists)

    y_true_bin = mlb.transform(y_true_lists)
    y_pred_bin = mlb.transform(y_pred_lists)

    print(f"\nTotal unique labels in evaluation (GT + prediction): {len(mlb.classes_)}")

    # Train vs Test label coverage
    train_label_series = (
        train_df["output_disease"]
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    train_label_series = train_label_series[train_label_series != ""]
    train_label_set = set(train_label_series.unique())

    test_label_set = set(label for sample in y_true_lists for label in sample)

    print("\nLabel-space comparison:")
    print(f"  TRAIN labels: {len(train_label_set)}")
    print(f"  TEST labels:  {len(test_label_set)}")
    print(f"  Test-only labels:  {len(test_label_set - train_label_set)}")
    print(f"  Train-only labels: {len(train_label_set - test_label_set)}")

    # Global metrics
    global_metrics = compute_global_metrics(y_true_bin, y_pred_bin, y_true_lists, y_pred_lists)

    # Per label metrics (per label F1 etc.)
    report_dict = classification_report(
        y_true_bin,
        y_pred_bin,
        target_names=mlb.classes_,
        zero_division=0,
        output_dict=True,
    )

    per_label_rows = []
    for label in mlb.classes_:
        if label in report_dict:
            stats = report_dict[label]
            per_label_rows.append({
                "metric_type": "per_label",
                "label": label,
                "precision": stats.get("precision", 0.0),
                "recall": stats.get("recall", 0.0),
                "f1": stats.get("f1-score", 0.0),
                "support": stats.get("support", 0),
            })

    # Train-based frequency regions
    minority, middle, majority = compute_regions_from_train(train_df)
    regions = {
        "minority": minority,
        "middle": middle,
        "majority": majority,
    }
    region_metrics = compute_region_metrics(y_true_bin, y_pred_bin, mlb, regions)

    # Per sample analysis
    sample_rows = []
    sample_jaccards = [
        jaccard_score([t], [p], average="samples") for t, p in zip(y_true_bin, y_pred_bin)
    ]

    for i, row in df.iterrows():
        case_id = row["case_id"] if "case_id" in row else i

        gold_labels = row["y_true"]   # sorted
        pred_labels = row["y_pred"]   # sorted

        set_gold = set(gold_labels)
        set_pred = set(pred_labels)

        num_gt = len(set_gold)
        num_pred = len(set_pred)
        diff_num = num_pred - num_gt

        fn_labels = sorted(list(set_gold - set_pred))
        fp_labels = sorted(list(set_pred - set_gold))

        # error type classification
        if num_gt == 0 and num_pred == 0:
            error_type = "true_none"
        elif num_gt == 0 and num_pred > 0:
            error_type = "spurious"
        elif num_gt > 0 and num_pred == 0:
            error_type = "miss_all"
        elif set_pred == set_gold:
            error_type = "exact"
        elif set_pred.issubset(set_gold):
            error_type = "under"
        elif set_gold.issubset(set_pred):
            error_type = "over"
        else:
            error_type = "mixed"

        # per sample over and under
        num_fp = len(fp_labels)
        num_fn = len(fn_labels)
        over_rate_sample = (num_fp / num_pred) if num_pred > 0 else 0.0
        under_rate_sample = (num_fn / num_gt) if num_gt > 0 else 0.0

        # BLEU like and ROUGE like (on sorted lists)
        bleu_val = label_bleu(pred_labels, gold_labels)
        rouge_val = label_rouge(pred_labels, gold_labels)

        sample_rows.append({
            "case_id": case_id,
            "input_finding": row.get("input_finding", ""),
            "output_diseases": ", ".join(gold_labels),
            "predicted_disease": ", ".join(pred_labels),
            "jaccard_score": sample_jaccards[i],
            "exact_match": int(set_pred == set_gold),
            "num_gt_labels": num_gt,
            "num_pred_labels": num_pred,
            "diff_num_labels": diff_num,
            "error_type": error_type,
            "num_fn_labels": num_fn,
            "num_fp_labels": num_fp,
            "over_pred_rate_sample": over_rate_sample,
            "under_pred_rate_sample": under_rate_sample,
            "fn_labels": ", ".join(fn_labels),
            "fp_labels": ", ".join(fp_labels),
            "bleu_label": bleu_val,
            "rouge_label": rouge_val,
        })

    # Overall BLEU and ROUGE (mean of per sample)
    overall_bleu = float(np.mean([r["bleu_label"] for r in sample_rows]))
    overall_rouge = float(np.mean([r["rouge_label"] for r in sample_rows]))

    print(f"\nOverall BLEU-label:  {overall_bleu:.4f}")
    print(f"Overall ROUGE-label: {overall_rouge:.4f}")

    # Cardinality based analysis (by number of true labels)
    cardinality_metrics = compute_cardinality_metrics(sample_rows)

    # Save CSVs with clean names
    base = os.path.splitext(os.path.basename(prediction_csv))[0]
    dir_name = os.path.dirname(prediction_csv)

    results_csv = os.path.join(dir_name, f"{base}_results.csv")
    metrics_csv = os.path.join(dir_name, f"{base}_metrics.csv")

    # Per sample analysis CSV
    pd.DataFrame(sample_rows).to_csv(results_csv, index=False)
    print(f"\nSaved detailed per sample analysis to: {results_csv}")

    # Metrics CSV (global + per label + region + BLEU/ROUGE + cardinality)
    metrics_rows = []

    # global metrics
    for k, v in global_metrics.items():
        metrics_rows.append({
            "metric_type": "global",
            "name": k,
            "value": v,
        })

    # add overall BLEU/ROUGE as global metrics
    metrics_rows.append({
        "metric_type": "global",
        "name": "overall_bleu_label",
        "value": overall_bleu,
    })
    metrics_rows.append({
        "metric_type": "global",
        "name": "overall_rouge_label",
        "value": overall_rouge,
    })

    # per label metrics (F1 etc.)
    metrics_rows.extend(per_label_rows)

    # region metrics
    for rm in region_metrics:
        metrics_rows.append({
            "metric_type": f"region_{rm['region']}",
            "name": "micro_f1",
            "value": rm["micro_f1"],
        })
        metrics_rows.append({
            "metric_type": f"region_{rm['region']}",
            "name": "macro_f1",
            "value": rm["macro_f1"],
        })

    # cardinality metrics (per number of true labels)
    for cm in cardinality_metrics:
        prefix = f"cardinality_num_gt_{cm['num_gt_labels']}"
        for name, value in cm["metrics"].items():
            metrics_rows.append({
                "metric_type": prefix,
                "name": name,
                "value": value,
            })

    pd.DataFrame(metrics_rows).to_csv(metrics_csv, index=False)
    print(f"Saved metrics summary to: {metrics_csv}")


# -----------------------------
# 7. ENTRY POINT
# -----------------------------
if __name__ == "__main__":

    PREDICTION_CSV = OUTPUT_DIR + ".csv"
    evaluate_multilabel_predictions(PREDICTION_CSV, TRAIN_CSV_PATH)


Loading prediction file: /content/drive/MyDrive/qwen3_4b_pretrained_eval-task-specific-prompt.csv
Loading train file: /content/drive/MyDrive/train.csv

Total unique labels in evaluation (GT + prediction): 166

Label-space comparison:
  TRAIN labels: 84
  TEST labels:  68
  Test-only labels:  19
  Train-only labels: 35

       PERFORMANCE METRICS
Exact Match Accuracy:  0.0518
Jaccard Score (IoU):   0.2661
----------------------------------------
Micro Precision:       0.2946
Micro Recall:          0.4597
Micro F1 Score:        0.3591
----------------------------------------
Macro Precision:       0.1150
Macro Recall:          0.1212
Macro F1 Score:        0.0972
----------------------------------------
Hamming Loss:          0.0254
Over prediction rate:  0.7054
Under prediction rate: 0.5403
Mean |#pred-#true|:    1.6943

TRAIN label frequency regions:
  Minority labels: 28
  Middle labels:   28
  Majority labels: 28

 PERFORMANCE BY LABEL FREQUENCY REGION (TRAIN BASED)

[MINORITY] (19 l